# omnicontext_gen_sp_iii_step2000 — the 400-row benchmark, variant (iii) at the selected checkpoint
Derived from `soft_prompt_test_gen_iii` by swapping the data source only. Variant (iii): learnable K/V injected at every MLLM layer via `past_key_values` — a layerwise K/V prefix-tuning implementation closely aligned with the mechanism Petrov's Section 4 analyses, without asserting parameterisation identity, since Petrov derives K/V from prefix vectors through the frozen projections while this learns them directly. The only parameter-matched point against LoRA rank 4.

Step 2000 is the argmax of GPT-4.1 val Overall over the post-warmup checkpoints on the domain val slice. The checkpoint is not reselected here: OmniContext measures what the domain-selected prompt does to general ability, so reselecting on this benchmark would answer a different question.

Sampler, negative prompt, reference handling and instruction handling are the domain notebook's, unchanged; the learned prompt is the only difference from baseline. Base weights are untouched.

Reads `omnicontext/manifest.jsonl` and `omnicontext/images/` at the pinned revision. Output: `gen/omnicontext/sp_iii_p139__step2000/{i:05d}.png`. Resumable.

Cost: 400 rows at roughly 20 s/img, near 2.3 h.


## 1. Repo

In [ ]:
import os, subprocess, sys
REPO = "/content/OmniGen2"
COMMIT = "18e6f9d5271b517fcb32e999f10df943ae9b8f20"   # the tree every arm of this thesis ran on
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
    subprocess.run(["git","-C",REPO,"checkout","--quiet",COMMIT], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], cwd=REPO, check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-U","huggingface_hub[hf_xet]>=0.30.0,<1.0"], check=True)
    print("installed.")
else:
    print("OmniGen2 already present.")

_head = subprocess.run(["git","-C",REPO,"rev-parse","HEAD"],
                       capture_output=True, text=True).stdout.strip()
print(_head[:7], "= repo commit")
# Everything this notebook asserts about encode_prompt, the three CFG branches and the rope tables
# was read off this tree. A moved upstream HEAD would change them while every check below still
# passed, so the commit is pinned rather than reported.
assert _head == COMMIT, (
    f"repo is at {_head[:7]}, expected {COMMIT[:7]}. Run: git -C {REPO} checkout {COMMIT}")
sys.path.insert(0, REPO)
os.chdir(REPO)

installed.
18e6f9d = repo commit


## 2. Drive, run identity, prompt checkpoint

In [ ]:
from google.colab import drive, userdata
import os
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

DRIVE   = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
DATASET = "omnicontext"         # 400-row public benchmark, 8 subtasks x 50
VARIANT = "iii"
RUN     = "sp_iii_p139"
P       = 139
KEY     = "prefix"               # tensor key inside prompt-<step>.pt
STEP    = 2000                # argmax over the post-warmup checkpoints (1000/2000/3000)
STEPS   = [STEP]                # kept as a list so the generation loop matches the val notebooks
EXP     = f"{DRIVE}/soft_prompt/experiments/{RUN}"

def ckpt_path(step):
    return f"{EXP}/prompt-{step}.pt"

assert os.path.exists(ckpt_path(STEP)), f"checkpoint missing: {ckpt_path(STEP)}"
print(f"{RUN}: selected checkpoint {STEP} present in {EXP}")

Mounted at /content/drive
sp_iii_p139: selected checkpoint 2000 present in /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/soft_prompt/experiments/sp_iii_p139


## 3. Images, split

In [ ]:
import subprocess, json
OC       = f"{DRIVE}/omnicontext"
TAR      = f"{OC}/images.tar"
LOCALTAR = "/content/oc_images.tar"
LOCAL    = "/content/oc"
IMGS     = f"{LOCAL}/images"

rows     = [json.loads(l) for l in open(f"{OC}/manifest.jsonl", encoding="utf-8")]
REVISION = open(f"{OC}/revision.txt").read().strip()
EXPECTED = sum(len(r["input_images"]) for r in rows)

def n_files(d): return sum(len(f) for _,_,f in os.walk(d)) if os.path.isdir(d) else 0

if os.path.exists(TAR):
    subprocess.run(["cp", TAR, LOCALTAR], check=True)
else:
    # tar exits non-zero writing into the Drive FUSE mount; build locally, copy the one file up
    _r = subprocess.run(["tar", "-cf", LOCALTAR, "-C", OC, "images"], capture_output=True, text=True)
    assert os.path.exists(LOCALTAR), _r.stderr[-800:]
    subprocess.run(["cp", LOCALTAR, TAR], check=True)
if n_files(IMGS) < EXPECTED:
    os.makedirs(LOCAL, exist_ok=True)
    subprocess.run(["tar", "-xf", LOCALTAR, "-C", LOCAL], check=True)

assert n_files(IMGS) == EXPECTED, (n_files(IMGS), EXPECTED)
missing = [p for r in rows for p in r["input_images"] if not os.path.exists(os.path.join(IMGS, p))]
assert not missing, f"{len(missing)} reference images missing, first: {missing[:3]}"
print(f"{DATASET}: {len(rows)} rows, all reference images present | revision {REVISION[:12]}")


omnicontext: 400 rows, all reference images present | revision 0f6aabbe8d34


## 4. Engine

In [ ]:
from PIL import Image, ImageOps
import torch

# shipped default from inference.py; quality filter, not a domain negative
NEG = ("(((deformed))), blurry, over saturation, bad anatomy, disfigured, poorly drawn face, mutation, mutated, "
       "(extra_limb), (ugly), (poorly drawn hands), fused fingers, messy drawing, broken legs censor, censored, censor_bar")

MAX_SEQ = 1024

def load_inputs(paths, max_side=512):   # refs downscaled; output size fixed at 512 via align_res=False
    out = []
    for p in paths:
        im = ImageOps.exif_transpose(Image.open(p).convert("RGB"))
        w, h = im.size
        s = max_side / max(w, h)
        if s < 1: w, h = int(w*s), int(h*s)
        w, h = max(16, (w//16)*16), max(16, (h//16)*16)
        out.append(im.resize((w, h), Image.LANCZOS))
    return out

## 5. Pipeline and rope tables
Base weights, no adapter.

In [ ]:
import torch, gc
from accelerate import Accelerator
from omnigen2.pipelines.omnigen2.pipeline_omnigen2 import OmniGen2Pipeline
from omnigen2.models.transformers.transformer_omnigen2 import OmniGen2Transformer2DModel

MODEL = "OmniGen2/OmniGen2"
accelerator = Accelerator(mixed_precision="bf16")
dtype = torch.bfloat16
DEV = accelerator.device

# base pipeline, unmodified weights: this arm trains no model parameter, only the prompt tensor
pipe = OmniGen2Pipeline.from_pretrained(MODEL, torch_dtype=dtype, trust_remote_code=True)
pipe.transformer = OmniGen2Transformer2DModel.from_pretrained(MODEL, subfolder="transformer", torch_dtype=dtype)
pipe = pipe.to(DEV)
print("pipeline ready on", DEV)

# Variant (iii) adds no conditioning rows: the prefix is attended to through past_key_values but
# emits no output, so cap_seq_len is the instruction length and the released axis still covers it.
_axes_before = tuple(pipe.transformer.config.axes_lens)
print(f"rope axes_lens {_axes_before} (unchanged: variant (iii) adds no conditioning rows)")

TOK = pipe.processor.tokenizer

def tokenize(prompt_text):
    """Chat-template then tokenize exactly as _get_qwen2_prompt_embeds does."""
    templated = pipe._apply_chat_template(prompt_text)
    ti = TOK([templated], padding="longest", max_length=MAX_SEQ, truncation=True, return_tensors="pt")
    return ti.input_ids.to(DEV), ti.attention_mask.to(DEV)

def _read(out):
    """train.py read .last_hidden_state off Qwen2_5_VLModel; the pipeline reads .hidden_states[-1]
    off Qwen2_5_VLForConditionalGeneration. Reads whichever the given class exposes so the two can
    be compared directly."""
    lhs = getattr(out, "last_hidden_state", None)
    return lhs if lhs is not None else out.hidden_states[-1]

@torch.no_grad()
def encode_stock(prompt_text, mllm=None):
    """The pipeline's own text path, with no prompt injected."""
    m = mllm if mllm is not None else pipe.mllm
    ids, mask = tokenize(prompt_text)
    feats = _read(m(input_ids=ids, attention_mask=mask, output_hidden_states=True))
    return feats.to(dtype=pipe.mllm.dtype), mask

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/content/OmniGen2/omnigen2/models/attention_processor.py:33: UserWarning: Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance")
/content/OmniGen2/omnigen2/models/transformers/block_lumina2.py:37: UserWarning: Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance")


model_index.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

mllm/model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Expected types for transformer: (<class 'omnigen2.models.transformers.transformer_omnigen2.OmniGen2Transformer2DModel'>,), got <class 'diffusers_modules.local.transformer_omnigen2.OmniGen2Transformer2DModel'>.


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

pipeline ready on cuda
rope axes_lens (1024, 1664, 1664) (unchanged: variant (iii) adds no conditioning rows)


## 6. Prompt injection and generation call
The injection mirrors the training patch line for line. CFG is positive-branch only: the learned rows enter through `g_txt * (pos - ref)` and are amplified 5x exactly as the instruction text is in every other arm; putting them in the negative branch would invert their meaning.

In [ ]:
from transformers.cache_utils import DynamicCache

# train.py builds the text encoder as `from transformers import Qwen2_5_VLModel as TextEncoder`, so
# the prefix was fitted to Qwen2_5_VLModel.forward. With a prefix cache and no explicit positions
# that path sets cache_position = arange(P, P + T) and position_ids from it, placing the text rows
# after the prefix on all three mRoPE axes. The pipeline holds the
# Qwen2_5_VLForConditionalGeneration wrapper, whose forward instead calls get_rope_index on the
# extended mask and caches self.rope_deltas on the module, so it disagrees with training and its
# result depends on whether it has been called before. The injected path therefore runs on the
# inner module the wrapper delegates to: same weights, checked in section 3 and section 8.
MLLM_INNER = getattr(pipe.mllm, "model", None)
assert type(MLLM_INNER).__name__ == "Qwen2_5_VLModel", (
    "inner encoder is %s, expected Qwen2_5_VLModel" % type(MLLM_INNER).__name__)

def _prefix_cache(B):
    pf = PREFIX.to(dtype=pipe.mllm.dtype)
    return DynamicCache.from_legacy_cache(tuple(
        (pf[l, 0].unsqueeze(0).expand(B, -1, -1, -1),
         pf[l, 1].unsqueeze(0).expand(B, -1, -1, -1))
        for l in range(pf.shape[0])))

@torch.no_grad()
def encode_injected(prompt_text, mllm=None):
    """Variant (iii): learnable K/V prepended at every layer. Mirrors the training patch — the
    prefix is attended to but emits no output rows, so text_feats keeps one row per input token
    and the mask handed downstream stays its original length."""
    m = mllm if mllm is not None else MLLM_INNER
    ids, mask = tokenize(prompt_text)
    B = ids.shape[0]
    mask_ext = torch.cat([torch.ones(B, PREFIX.shape[3], dtype=mask.dtype, device=mask.device),
                          mask], dim=1)
    feats = _read(m(input_ids=ids, attention_mask=mask_ext, past_key_values=_prefix_cache(B),
                    use_cache=False, output_hidden_states=True))
    assert feats.shape[1] == ids.shape[1], (
        "prefix must not add output rows: %d vs %d" % (feats.shape[1], ids.shape[1]))
    assert mask.shape[1] == feats.shape[1], (mask.shape, feats.shape)
    return feats.to(dtype=pipe.mllm.dtype), mask

INJECTION_NOTE = ("variant (iii): learnable K/V at every MLLM layer via past_key_values; "
                  "extended mask on the encoder call only, original mask passed downstream")

@torch.no_grad()
def generate_with(instruction, input_images=None, negative=NEG,
                  seed=0, steps=50, text_g=5.0, image_g=2.0, size=512):
    """Sampler settings identical to every other arm. The learned prompt enters through the
    POSITIVE branch only: encode_prompt skips _get_qwen2_prompt_embeds when prompt_embeds is
    supplied and encodes the negative prompt itself, so model_pred_ref and model_pred_uncond stay
    on the stock negative embeds. `prompt` is still passed because encode_prompt chat-templates it
    before reaching the prompt_embeds branch; the templated string is discarded."""
    pe, pm = encode_injected(instruction)
    gen = torch.Generator(device=DEV).manual_seed(seed)
    res = pipe(prompt=instruction,
               prompt_embeds=pe, prompt_attention_mask=pm,
               negative_prompt=negative,
               input_images=input_images,
               width=size, height=size, num_inference_steps=steps, max_sequence_length=MAX_SEQ,
               text_guidance_scale=text_g, image_guidance_scale=image_g, cfg_range=(0.0, 1.0),
               num_images_per_prompt=1, generator=gen, output_type="pil", align_res=False)
    return res.images[0]

## 7. Equivalence gate
8 checks, before generation. The same gate the domain notebooks ran, re-run over the 400 OmniContext rows: the two tokenizers agree on every row of the benchmark; the stock path reproduces the pipeline's own encoder; the wrapper agrees with the inner model; the injection has the effect it must and none it must not; shapes and mask lengths are what the variant requires; the rope table covers the longest row; and the pipeline MLLM is numerically the same encoder train.py fitted the prompt against.


In [ ]:
# Gate: prove the inference-side injection reproduces the training-side one, before 3.2 h of
# generation. Two things differ between the two sides and neither was exercised by the probes,
# which contain no inference code at all:
#   - train.py encoded text with transformers Qwen2_5_VLModel and read .last_hidden_state; the
#     pipeline holds a Qwen2_5_VLForConditionalGeneration and reads .hidden_states[-1].
#   - train.py tokenized with Qwen/Qwen2.5-VL-3B-Instruct; the pipeline uses its own processor
#     tokenizer. The chat template text is identical in both, but the token ids need not be.
import torch, transformers, gc
from transformers import AutoTokenizer
print("transformers", transformers.__version__)
# requirements.txt at this commit pins 4.51.3, and the probes recorded 4.51.3 when the mechanism
# was validated. The encoder forward and, for variant (iii), DynamicCache semantics are
# version-specific, and a drift would move the conditioning silently.
assert transformers.__version__ == "4.51.3", (
    f"transformers {transformers.__version__}; this arm was trained and probed on 4.51.3")

PREFIX = torch.load(ckpt_path(STEP), map_location="cpu")[KEY].to(DEV)
assert PREFIX.ndim == 5 and PREFIX.shape[3] == P, PREFIX.shape
assert PREFIX.float().norm().item() > 0, "the loaded prefix is all zeros"

txt = rows[0]["instruction"]
ids, mask = tokenize(txt)

INNER = getattr(pipe.mllm, "model", None)
assert INNER is not None, ("inner model not at pipe.mllm.model; available: "
                           + str([a for a in dir(pipe.mllm) if a in ("model", "language_model", "base_model")]))
print(f"wrapper {type(pipe.mllm).__name__} | inner {type(INNER).__name__}")

print()
print("=== 1. tokenizer parity over the whole split: pipeline processor vs the training tokenizer ===")
# every row, not just one: a single row cannot show that the two tokenizers agree on the split
SYS = "You are a helpful assistant that generates high-quality images based on user instructions."
_tt = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")
_bad_tpl, _bad_ids = [], []
for _j, _r in enumerate(rows):
    _t1 = _tt.apply_chat_template([{"role": "system", "content": SYS},
                                   {"role": "user", "content": _r["instruction"]}],
                                  tokenize=False, add_generation_prompt=False)
    _t2 = pipe._apply_chat_template(_r["instruction"])
    if _t1 != _t2:
        _bad_tpl.append(_j)
        continue
    _x = _tt(_t1, return_tensors="pt").input_ids
    _y = TOK(_t2, return_tensors="pt").input_ids
    if _x.shape != _y.shape or not bool((_x == _y).all()):
        _bad_ids.append(_j)
print(f"  chat template identical on {len(rows) - len(_bad_tpl)}/{len(rows)} rows")
print(f"  token ids identical on      {len(rows) - len(_bad_tpl) - len(_bad_ids)}/{len(rows)} rows")
assert not _bad_tpl, f"chat templates differ between training and inference on rows {_bad_tpl[:5]}"
assert not _bad_ids, f"the two tokenizers disagree on rows {_bad_ids[:5]}"

print()
print("=== 2. stock path reproduces the pipeline's own encoder ===")
_pe, _pm = pipe._get_qwen2_prompt_embeds(prompt=[pipe._apply_chat_template(txt)],
                                         device=DEV, max_sequence_length=MAX_SEQ)
_me, _mm = encode_stock(txt)
d = (_pe.float() - _me.float()).abs().max().item()
print(f"  max |mine - _get_qwen2_prompt_embeds| = {d:.3e}  shapes {tuple(_me.shape)} vs {tuple(_pe.shape)}")
assert _me.shape == _pe.shape and _mm.shape == _pm.shape, (_me.shape, _pe.shape)
assert d <= 1e-3, "the stock reimplementation diverges from the pipeline's own text path"

print()
print("=== 3. stock path: wrapper vs inner model ===")
s_w, sm = encode_stock(txt)
s_i, _ = encode_stock(txt, mllm=INNER)
d = (s_w.float() - s_i.float()).abs().max().item()
print(f"  max |wrapper - inner| = {d:.3e}")
assert d <= 1e-3, "wrapper and inner disagree on the stock path"

print()
print("=== 4. injected path runs on the encoder class the prefix was fitted to ===")
# Unlike variants (i) and (ii), this one hands the encoder an attention mask longer than the input,
# and the two classes resolve positions differently: the wrapper reads get_rope_index off the
# extended mask and caches rope_deltas, the inner model reads arange(P, P + T) off the prefix cache.
# Only the inner one is what train.py ran, so the gap is recorded, not asserted away. Section 3
# already showed the two agree exactly when no prefix is passed, so this is the injection path only.
f_w, fm = encode_injected(txt)
f_x, _ = encode_injected(txt, mllm=pipe.mllm)
d = (f_x.float() - f_w.float()).abs().max().item()
print(f"  default encoder      : {type(MLLM_INNER).__name__}   (train.py: Qwen2_5_VLModel)")
print(f"  wrapper vs inner     : {d:.3e}   (expected to differ; the wrapper is not the trained path)")
assert type(MLLM_INNER).__name__ == "Qwen2_5_VLModel", type(MLLM_INNER).__name__
del f_x

# no P=0 step for variant (iii): a zero-length DynamicCache is an edge case in transformers and a
# spurious failure there would say nothing. Check 3 already compares the injected path across the
# two encoder classes, which is the property that matters.

print()
print("=== 5. effect: the prefix must change the conditioning ===")
# The decisive check for this variant. feats.shape[1] == ids.shape[1] holds whether the cache is
# used or silently discarded, so shape alone cannot tell the two apart: if transformers ignored
# past_key_values under use_cache=False, generation would run with no prefix and produce baseline
# images with every other assertion still passing.
eff = (f_w.float() - s_w.float()).abs().max().item()
print(f"  max |injected - stock| = {eff:.3e}")
assert f_w.shape == s_w.shape, (f_w.shape, s_w.shape)
assert eff > 0, ("past_key_values had no effect: the prefix was discarded, so generation would "
                 "reproduce the baseline. Check the transformers version against the trainer run.")

print()
print("=== 6. shapes and mask, three rows ===")
for _r in rows[:3]:
    _i, _ = tokenize(_r["instruction"])
    _f, _m = encode_injected(_r["instruction"])
    print(f"  instruction rows {_i.shape[1]:>4} | conditioning rows {_f.shape[1]:>4} | mask {_m.shape[1]:>4}")
    assert _m.shape[1] == _f.shape[1], (_m.shape, _f.shape)
    assert _f.shape[1] == _i.shape[1], (_f.shape, _i.shape)

print()
print("=== 7. rope and sequence-length headroom over the whole split ===")
# measured WITHOUT truncation: a truncated row would report exactly MAX_SEQ and pass a <= test
lens = [TOK(pipe._apply_chat_template(r["instruction"]), return_tensors="pt").input_ids.shape[1]
        for r in rows]
cap = max(lens)
print(f"  templated instruction (untruncated): mean {sum(lens)/len(lens):.0f}, max {max(lens)}")
print(f"  max cap_seq_len = {cap} | released axis {_axes_before[0]} (not extended)")
assert cap <= _axes_before[0], "instruction overflows the released rope bound; variant (iii) must not"
assert max(lens) < MAX_SEQ, "an instruction reaches max_sequence_length and would be truncated"
print(f"  max_sequence_length headroom: {MAX_SEQ - max(lens)} tokens, 0 rows truncated")

print()
print("=== 8. encoder checkpoint parity: pipeline MLLM vs the encoder training fitted the prompt to ===")
# The one remaining way these images could be silently wrong. train.py builds text_encoder from
# options/ft_lora.yml's pretrained_text_encoder_model_name_or_path = "Qwen/Qwen2.5-VL-3B-Instruct",
# while the pipeline's mllm is loaded from the OmniGen2/OmniGen2 repo, whose MLLM carries a trained
# "<|img|>" token on top of Qwen. Same architecture, different checkpoints. If any weight the
# instruction touches differs, the prompt is deployed against an encoder it was never fitted to and
# every image is quietly degraded while every check above still passes. Tokenizer parity does not
# cover this: an added token leaves the ids for ordinary text unchanged.
from transformers import Qwen2_5_VLModel
_train_enc = Qwen2_5_VLModel.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct", torch_dtype=dtype).to(DEV).eval()
_e_pipe = pipe.mllm.get_input_embeddings()(ids)
_e_train = _train_enc.get_input_embeddings()(ids)
d_emb = (_e_pipe.float() - _e_train.float()).abs().max().item()
_t_stock, _ = encode_stock(txt, mllm=_train_enc)
d_stock = (_t_stock.float() - s_w.float()).abs().max().item()
_t_inj, _ = encode_injected(txt, mllm=_train_enc)
d_inj = (_t_inj.float() - f_w.float()).abs().max().item()
print(f"  vocab size: pipeline {pipe.mllm.config.vocab_size} | training {_train_enc.config.vocab_size}")
print(f"  max |input-embedding| difference on these ids = {d_emb:.3e}")
print(f"  max |stock conditioning| difference           = {d_stock:.3e}")
print(f"  max |injected conditioning| difference        = {d_inj:.3e}")
del _train_enc, _t_stock, _t_inj, _e_pipe, _e_train
gc.collect(); torch.cuda.empty_cache()
assert max(d_emb, d_stock, d_inj) <= 1e-3, (
    "the pipeline MLLM and the training MLLM produce different conditioning: the prompt was fitted "
    "to one checkpoint and would be deployed on the other. Do not generate.")
print("  the two checkpoints agree on this input: the prompt runs on what it was fitted to")

print()
print("[PASS] tokenizer, encoder checkpoint, encoder class and injection all match the training path.")

transformers 4.51.3
wrapper Qwen2_5_VLForConditionalGeneration | inner Qwen2_5_VLModel

=== 1. tokenizer parity over the whole split: pipeline processor vs the training tokenizer ===


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.53G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

  vocab size: pipeline 151936 | training 151936
  max |input-embedding| difference on these ids = 0.000e+00
  max |stock conditioning| difference           = 0.000e+00
  max |injected conditioning| difference        = 0.000e+00
  the two checkpoints agree on this input: the prompt runs on what it was fitted to

[PASS] tokenizer, encoder checkpoint, encoder class and injection all match the training path.


## 8. Generate (resumable)

In [ ]:
import time, gc

def gen_step(step):
    global PREFIX
    GEN = f"{DRIVE}/gen/{DATASET}/{RUN}__step{step}"
    os.makedirs(GEN, exist_ok=True)
    have = {int(f[:-4]) for f in os.listdir(GEN) if f.endswith(".png") and f[:-4].isdigit()}
    todo = [i for i in range(len(rows)) if i not in have]
    print(f"\n=== step {step}: {len(have)} done, {len(todo)} to generate -> {GEN} ===")
    if not todo:
        return
    PREFIX = torch.load(ckpt_path(step), map_location="cpu")[KEY].to(DEV)
    assert PREFIX.ndim == 5 and PREFIX.shape[3] == P, PREFIX.shape
    assert PREFIX.float().norm().item() > 0, "the loaded prefix is all zeros"
    t0 = time.time()
    for k, i in enumerate(todo):
        r = rows[i]
        refs = load_inputs([os.path.join(IMGS, p) for p in r["input_images"]])
        img = generate_with(r["instruction"], input_images=refs or None)
        assert img.size == (512, 512), (i, img.size)
        img.save(f"{GEN}/{i:05d}.png")
        if (k + 1) % 25 == 0:
            print(f"  {k+1}/{len(todo)}  ({(time.time()-t0)/(k+1):.1f}s/img)")
    n = len([f for f in os.listdir(GEN) if f.endswith(".png")])
    print(f"  step {step}: {n}/{len(rows)} images")
    assert n == len(rows), "incomplete"

for step in STEPS:
    gen_step(step)
gc.collect(); torch.cuda.empty_cache()
print("\ndone")


=== step 2000: 0 done, 400 to generate -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/omnicontext/sp_iii_p139__step2000 ===


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

...

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  400/400  (22.4s/img)
  step 2000: 400/400 images

done


## 9. Manifest

In [ ]:
import json, datetime, torch

for step in STEPS:
    GEN = f"{DRIVE}/gen/{DATASET}/{RUN}__step{step}"
    n = len([f for f in os.listdir(GEN) if f.endswith(".png")]) if os.path.isdir(GEN) else 0
    if n == 0:
        continue
    man = {"dataset": DATASET, "run": RUN, "variant": VARIANT, "step": step, "P": P, "n_images": n,
           "dataset_revision": REVISION,
           "selection": "argmax val Overall over 500/1000/2000/3000, pre-registered",
           "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
           "prompt_checkpoint": ckpt_path(step),
           "injection": INJECTION_NOTE,
           "cfg_branch": "positive only: prompt_embeds passed precomputed, negative branch encoded by the pipeline",
           "rope_axes_lens": list(pipe.transformer.config.axes_lens),
           "sampler": {"steps": 50, "text_guidance_scale": 5.0, "image_guidance_scale": 2.0,
                       "cfg_range": [0.0, 1.0], "max_sequence_length": MAX_SEQ, "seed": 0,
                       "size": 512, "align_res": False},
           "negative_prompt": NEG,
           "written": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
    json.dump(man, open(f"{GEN}/manifest.json", "w"), indent=2)
print("manifest written")

manifest written
